# 01 - Malimg Veri Seti İnceleme

**Amaç:** Veri setini tanımak — kaç aile var, her ailede kaç örnek var, sınıflar ne kadar dengesiz, görüntülerin boyutları neler ve aileler görsel olarak birbirinden nasıl ayrışıyor.

Bu notebook'un ürettiği grafikler `reports/figures/` altına kaydedilir ve rapora doğrudan eklenebilir.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data_loader import load_dataset_index, class_distribution, image_stats, load_image

sns.set_theme(style="whitegrid")
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)

## 1. Veri setini listeleme

In [ ]:
df = load_dataset_index(config.MALIMG_DIR)
print(f"Toplam görüntü : {len(df)}")
print(f"Aile sayısı    : {df['family'].nunique()}")
df.head()

## 2. Sınıf dağılımı

Malimg dengesiz bir veri setidir; bu tablo hangi ailelerin az örneği olduğunu gösterir. `class_weight="balanced"` kararının gerekçesi buradan çıkacak.

In [ ]:
dist = class_distribution(df)
dist

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
sns.barplot(x=dist["adet"], y=dist.index, ax=ax, color="steelblue")
for i, v in enumerate(dist["adet"]):
    ax.text(v + dist["adet"].max() * 0.01, i, str(v), va="center", fontsize=8)
ax.set_xlabel("Örnek sayısı")
ax.set_ylabel("Zararlı yazılım ailesi")
ax.set_title("Malimg - aile başına örnek sayısı")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "01_sinif_dagilimi.png", dpi=150)
plt.show()

In [ ]:
en_cok, en_az = dist["adet"].max(), dist["adet"].min()
print(f"En kalabalık aile : {dist.index[0]} ({en_cok})")
print(f"En az örnekli aile: {dist.index[-1]} ({en_az})")
print(f"Dengesizlik oranı : {en_cok / en_az:.1f} kat")
print(f"İlk 3 ailenin payı: %{dist['yuzde'].iloc[:3].sum():.1f}")

## 3. Görüntü boyutları

Malimg görüntülerinin genişliği orijinal dosya boyutuna göre değişir, yüksekliği de buna bağlıdır. Bu yüzden ön işlemede hepsini 256×256'ya getireceğiz.

In [ ]:
stats = image_stats(df)  # 9.339 görüntü için birkaç saniye sürer
stats[["width", "height", "file_kb"]].describe().round(1)

In [ ]:
print("Görüntü modları:", stats["mode"].value_counts().to_dict())
print("\nGenişlik değerleri:")
print(stats["width"].value_counts().sort_index().to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(stats["height"], bins=50, ax=axes[0])
axes[0].set_title("Yükseklik dağılımı (piksel)")
sns.boxplot(data=stats, x="file_kb", y="family", ax=axes[1], fliersize=1,
            order=dist.index)
axes[1].set_title("Aileye göre dosya boyutu (KB)")
axes[1].tick_params(axis="y", labelsize=7)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "01_boyutlar.png", dpi=150)
plt.show()

## 4. Her aileden örnek görüntü

Aynı ailenin görüntüleri benzer doku desenleri taşır; LBP ve GLCM'nin yakalamaya çalışacağı şey bu.

In [ ]:
families = dist.index.tolist()
cols = 5
rows = int(np.ceil(len(families) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.6, rows * 2.8))
for ax, fam in zip(axes.flat, families):
    path = df[df["family"] == fam]["path"].iloc[0]
    ax.imshow(load_image(path), cmap="gray", aspect="auto")
    ax.set_title(fam, fontsize=8)
    ax.axis("off")
for ax in axes.flat[len(families):]:
    ax.axis("off")
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "01_ornek_goruntuler.png", dpi=150)
plt.show()

## 5. Aynı aile içindeki benzerlik

En kalabalık aileden birkaç örnek yan yana; aile içi benzerlik yüksekse sınıflandırma kolaylaşır.

In [ ]:
fam = dist.index[0]
paths = df[df["family"] == fam]["path"].sample(5, random_state=config.RANDOM_STATE)
fig, axes = plt.subplots(1, 5, figsize=(13, 3))
for ax, p in zip(axes, paths):
    ax.imshow(load_image(p), cmap="gray", aspect="auto")
    ax.axis("off")
fig.suptitle(f"{fam} ailesinden 5 örnek")
plt.tight_layout()
plt.show()

## 6. Piksel yoğunluğu dağılımı

Ailelerin byte (0-255) dağılımları farklıysa bu, özellik çıkarımında ayırt edici bilgi olduğunun ilk işaretidir.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
for fam in dist.index[:6]:
    sample = df[df["family"] == fam]["path"].sample(
        min(20, (df["family"] == fam).sum()), random_state=config.RANDOM_STATE)
    pixels = np.concatenate([load_image(p).ravel() for p in sample])
    hist, _ = np.histogram(pixels, bins=64, range=(0, 256), density=True)
    ax.plot(np.linspace(0, 255, 64), hist, label=fam)
ax.set_yscale("log")
ax.set_xlabel("Piksel değeri (byte)")
ax.set_ylabel("Yoğunluk (log)")
ax.set_title("En kalabalık 6 ailenin piksel dağılımı")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(config.FIGURES_DIR / "01_piksel_dagilimi.png", dpi=150)
plt.show()

## 7. Notlar (rapora girecek gözlemler)

Notebook'u gerçek veriyle çalıştırdıktan sonra buraya yazın:

- Toplam görüntü / aile sayısı:
- Dengesizlik oranı ve en az örnekli aileler:
- Görüntü boyutlarıyla ilgili gözlem:
- Görsel olarak birbirine benzeyen aileler (ileride karışabilecekler):